# A2 Q5 - Codabench Submissions for the Two-Stage Pipeline

Produces the leaderboard submission for the full two-stage pipeline on both
blind populations: Stage-1 BM25 + frozen-embedding `score_inview` scores over
each impression's `article_ids_inview`, joined to Q1's behavioural features,
scored by the Q2 LightGBM re-ranker, and written as ranks in each
competition's exact format (`predictions.txt` / `prediction.txt` in a zip,
same as A1's `ebnerd_testset_submission.ipynb` / `mind_large_test_submission.ipynb`).

Inputs, per population (SPEC.md A2 Q5 #6):

- `data/processed/{population}/behaviors.parquet` + `history.parquet` -- the
  converted blind test sets from A1 Q5;
- `data/processed/{population}/reranker_features.parquet` -- built by
  `feature_engineering.py` with `FEATURE_DATASETS={population}`, in
  `(user_id, impression_id)` order;
- the training dataset's `reranker_model_*.txt` + metadata, and for
  `ebnerd_testset` its catalog and embeddings (`reranker.SUBMISSION_POPULATIONS`).

The feature table is never joined against the Stage-1 scores: both are
produced over the same user-sorted order, so the notebook walks the feature
table in impression-aligned slices, scores Stage-1 for exactly those
impressions, and checks per chunk that the two sides list the same
impressions and candidates in the same order. Chunked and checkpointed like
every long local loop in this project. Run with
`SUBMISSION_DATASETS=mind_large_test uv run python reranker_submission.py`
(one population per kernel).

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import zipfile

import lightgbm as lgb
import numpy as np
import polars as pl

from cs4406m26_assignment1c1.reranker import (
    FEATURE_COLUMNS,
    SUBMISSION_POPULATIONS,
    feature_matrix,
    write_user_sorted_source,
)
from cs4406m26_assignment1c1.retrieval import build_stage1_scorers


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
SUBMISSIONS_DIR = ROOT / "submissions"
PROGRESS_LOG = ROOT / "build_progress.log"
CHECKPOINT_DIR = DATA_DIR / "_submission_checkpoints"

# Impressions per checkpointed chunk. 200,000 impressions is ~3M candidate
# rows on ebnerd_testset and ~8M on mind_large_test (39 candidates per
# impression): one feature slice, one Stage-1 pass and one booster.predict
# per chunk, each freed before the next.
CHUNK_IMPRESSIONS = 200_000
# Lines per slice when streaming the prediction file out (never the whole
# 13.5M-line file as one Python string).
WRITE_ROWS = 500_000

# Serving-time value for the one feature the blind sets cannot carry.
# clicks_earlier_in_session counts the user's clicks in EARLIER impressions of
# the same session -- known to a live system, but the leaderboard test sets
# ship no click labels at all, so it is null in their feature table. 0 is its
# modal training value (45.6% of ebnerd_large training rows) and is also
# exactly what LightGBM substitutes for NaN in a feature it never saw missing
# during training -- test_feature_assembly checks that equivalence on real
# rows. The feature carries 0.3% of the EB-NeRD booster's gain and 0% of MIND's.
CLICKS_EARLIER_IMPUTE = 0

# The per-row key. impression_id alone is not unique on ebnerd_testset: its
# 200,000 beyond-accuracy rows all carry the sentinel impression_id 0 (one
# row per user, one shared 250-article candidate list -- A1 SPEC.md Q5 #8).
# Every join and uniqueness check here is on (impression_id, user_id), which
# is unique on both populations.
ROW_KEY = ["impression_id", "user_id"]

_env = os.environ.get("SUBMISSION_DATASETS")
DATASETS = _env.split(",") if _env else list(SUBMISSION_POPULATIONS)
for _d in DATASETS:
    if _d not in SUBMISSION_POPULATIONS:
        raise ValueError(f"{_d} is not a submission population; choose from {list(SUBMISSION_POPULATIONS)}")


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] reranker_submission: {message}\n")
        f.flush()


def sink_parquet_atomic(lf: pl.LazyFrame, path: Path) -> None:
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    lf.sink_parquet(tmp_path)
    os.replace(tmp_path, path)


log_progress(f"reranker_submission started (datasets={DATASETS})")

store = {}
for name in DATASETS:
    cfg = SUBMISSION_POPULATIONS[name]
    train, catalog = cfg["train_dataset"], cfg["catalog_dataset"]
    model_path = DATA_DIR / train / f"reranker_model_{train}.txt"
    meta_path = DATA_DIR / train / f"reranker_metadata_{train}.json"
    features_path = DATA_DIR / name / "reranker_features.parquet"
    feature_meta_path = DATA_DIR / name / "feature_metrics.json"
    for p in (model_path, meta_path, features_path, feature_meta_path):
        if not p.exists():
            raise FileNotFoundError(f"missing {p} (see README.md, A2 Q5 Part B)")
    metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    if metadata["feature_columns"] != FEATURE_COLUMNS:
        raise ValueError(f"{name}: booster feature order {metadata['feature_columns']} != {FEATURE_COLUMNS}")
    feature_meta = json.loads(feature_meta_path.read_text(encoding="utf-8"))
    # The chunk alignment below is only valid over the user-sorted table.
    if feature_meta.get("row_order") != "user_id, impression_id":
        raise ValueError(f"{name}: reranker_features.parquet is not user-sorted (row_order={feature_meta.get('row_order')!r}); "
                         f"rebuild it with FEATURE_DATASETS={name} uv run python feature_engineering.py")
    if feature_meta.get("train_dataset") != train or feature_meta.get("has_click_labels") is not False:
        raise ValueError(f"{name}: feature_metrics.json does not describe a blind population built against {train}: {feature_meta}")
    store[name] = {
        "articles": pl.read_parquet(DATA_DIR / catalog / "articles.parquet", columns=["article_id", "title", "abstract"]),
        "history_path": DATA_DIR / name / "history.parquet",
        "embeddings_path": DATA_DIR / catalog / "article_embeddings.parquet",
        "behaviors_path": DATA_DIR / name / "behaviors.parquet",
        "features_path": features_path,
        "n_impressions": feature_meta["n_impressions_by_split"]["test"],
        "has_session_data": feature_meta["has_session_data"],
        "booster": lgb.Booster(model_file=str(model_path)),
        "metadata": metadata,
        "train_dataset": train,
        "prediction_filename": cfg["prediction_filename"],
        "strip_impression_prefix": cfg["strip_impression_prefix"],
    }
    (CHECKPOINT_DIR / name).mkdir(parents=True, exist_ok=True)
    (SUBMISSIONS_DIR / name).mkdir(parents=True, exist_ok=True)
    log_progress(f"  {name}: booster ({metadata['best_iteration']} trees, trained on {train}) + catalog ({store[name]['articles'].height} articles) loaded")

{name: {"impressions": store[name]["n_impressions"], "trees": store[name]["booster"].num_trees(), "trained_on": store[name]["train_dataset"]} for name in DATASETS}

{'mind_large_test': {'impressions': 2370727,
  'trees': 45,
  'trained_on': 'mind_large'}}

In [2]:
def test_setup():
    for name in DATASETS:
        s = store[name]
        assert s["booster"].num_feature() == len(FEATURE_COLUMNS)
        n_beh = pl.scan_parquet(s["behaviors_path"]).select(pl.len()).collect().item()
        assert n_beh == s["n_impressions"], (name, n_beh, s["n_impressions"])
        # No labels anywhere in the blind population -- the whole point of it.
        assert "article_ids_clicked" not in pl.scan_parquet(s["behaviors_path"]).collect_schema().names()
        # Every in-view article of the first rows resolves in the catalog the
        # scorers are built over (the full check is the per-chunk KeyError
        # the BM25 adapter would raise otherwise).
        head = pl.scan_parquet(s["behaviors_path"]).select("article_ids_inview").head(2_000).collect()
        catalog_ids = set(s["articles"]["article_id"].to_list())
        assert all(a in catalog_ids for ids in head["article_ids_inview"].to_list() for a in ids), name


test_setup()
print("ok: boosters match FEATURE_COLUMNS, populations are label-free and their in-view ids resolve in the serving catalog")

ok: boosters match FEATURE_COLUMNS, populations are label-free and their in-view ids resolve in the serving catalog


## Impression-aligned chunking

The feature table is in `(user_id, impression_id)` order with each
impression's candidates contiguous in `position_in_impression` order. Rebuilding
the same order from `behaviors.parquet` (the same deterministic external sort,
~1 minute) gives the per-impression candidate counts, hence the row offset of
every impression in the feature table, without reading the table's 200M-row
key column. Each chunk then cross-checks the two sides impression by
impression and candidate by candidate, so a misalignment fails loudly rather
than scoring one impression's features against another's retrieval scores.

In [3]:
def sorted_source_path(dataset: str) -> Path:
    return CHECKPOINT_DIR / dataset / "source_user_sorted.parquet"


def build_chunk_plan(dataset: str) -> dict:
    """`{"source": path, "imp_bounds": [...], "row_bounds": [...]}` where chunk
    c covers impressions imp_bounds[c]:imp_bounds[c+1] of the user-sorted
    order, which occupy feature-table rows row_bounds[c]:row_bounds[c+1]."""
    src = sorted_source_path(dataset)
    if not src.exists():
        n = write_user_sorted_source(store[dataset]["behaviors_path"], src, ["impression_id", "user_id", "article_ids_inview"])
        log_progress(f"  {dataset}: user-sorted source written ({n} impressions)")
    lengths = pl.scan_parquet(src).select(pl.col("article_ids_inview").list.len()).collect()["article_ids_inview"].to_numpy()
    n_imp = len(lengths)
    assert n_imp == store[dataset]["n_impressions"], (dataset, n_imp, store[dataset]["n_impressions"])
    row_offsets = np.concatenate([[0], np.cumsum(lengths, dtype=np.int64)])
    n_rows = pl.scan_parquet(store[dataset]["features_path"]).select(pl.len()).collect().item()
    if n_rows != int(row_offsets[-1]):
        raise ValueError(f"{dataset}: feature table has {n_rows} rows, behaviors imply {int(row_offsets[-1])}")
    imp_bounds = list(range(0, n_imp, CHUNK_IMPRESSIONS)) + [n_imp]
    return {"source": src, "imp_bounds": imp_bounds, "row_bounds": [int(row_offsets[i]) for i in imp_bounds]}


chunk_plans = {name: build_chunk_plan(name) for name in DATASETS}
{name: {"chunks": len(p["imp_bounds"]) - 1, "rows": p["row_bounds"][-1]} for name, p in chunk_plans.items()}

{'mind_large_test': {'chunks': 12, 'rows': 93115001}}

In [4]:
def test_chunk_plan():
    for name in DATASETS:
        plan = chunk_plans[name]
        assert plan["imp_bounds"][0] == 0 and plan["imp_bounds"][-1] == store[name]["n_impressions"]
        assert plan["row_bounds"][0] == 0 and all(b > a for a, b in zip(plan["row_bounds"], plan["row_bounds"][1:]))
        # First chunk: feature-table keys equal the sorted source's expansion,
        # row for row. This is the alignment every later chunk re-checks.
        s, e = plan["imp_bounds"][0], plan["imp_bounds"][1]
        r0, r1 = plan["row_bounds"][0], plan["row_bounds"][1]
        src = pl.scan_parquet(plan["source"]).slice(s, e - s).select("impression_id", "user_id", "article_ids_inview").collect()
        expanded = src.explode("article_ids_inview").rename({"article_ids_inview": "article_id"})
        feat = pl.scan_parquet(store[name]["features_path"]).slice(r0, r1 - r0).select("impression_id", "user_id", "article_id").collect()
        assert feat.equals(expanded), f"{name}: feature rows do not match the user-sorted source expansion"
        # and the source really is user-sorted with unique impressions
        keys = pl.scan_parquet(plan["source"]).select("user_id", "impression_id").collect()
        assert keys["user_id"].is_sorted() and keys.select(pl.struct(ROW_KEY).n_unique()).item() == keys.height


test_chunk_plan()
print("ok: chunk plan spans the whole population, first chunk's feature rows equal the sorted source expansion key-for-key")

C:\Users\HP\AppData\Local\Temp\ipykernel_10204\2741325010.py:11: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  expanded = src.explode("article_ids_inview").rename({"article_ids_inview": "article_id"})


ok: chunk plan spans the whole population, first chunk's feature rows equal the sorted source expansion key-for-key


## Stage-1 scoring, feature assembly and re-ranking (chunked, checkpointed)

Per chunk: the feature slice and the matching source slice are read, the
shared `build_stage1_scorers` adapters produce `bm25_score`/`embedding_score`
over each impression's in-view list (user-sorted, so the BM25 one-entry cache
hits ~17 times per user on EB-NeRD), the two retrieval-membership flags are
`False` (the corpus-wide top-200 lists of A1 exist for the labelled datasets'
users only; they carried zero split gain in both boosters, SPEC.md A2 Q2 #9),
`clicks_earlier_in_session` is imputed, and the booster scores the
`feature_matrix`. Ranks are stable-argsort descending, 1-based, per impression
-- the same formula A1's submission notebooks used.

In [5]:
_scorer_cache = {"dataset": None, "scorers": None}


def get_scorers(dataset: str) -> dict:
    if _scorer_cache["dataset"] != dataset:
        _scorer_cache["dataset"] = None
        _scorer_cache["scorers"] = None
        log_progress(f"  {dataset}: building stage-1 scorers")
        _scorer_cache["scorers"] = build_stage1_scorers(
            store[dataset]["articles"], store[dataset]["history_path"], store[dataset]["embeddings_path"]
        )
        _scorer_cache["dataset"] = dataset
        log_progress(f"  {dataset}: stage-1 scorers ready ({_scorer_cache['scorers']['n_docs']} docs)")
    return _scorer_cache["scorers"]


def stage1_columns(scorers: dict, user_ids: list, inviews: list) -> tuple[list, list]:
    """Flat bm25/embedding score columns over the impressions' candidates, in
    (impression, position) order -- the feature table's row order."""
    out_bm25, out_emb = [], []
    for uid, ids in zip(user_ids, inviews):
        ids = list(ids)
        b = scorers["bm25"](uid, ids)
        e = scorers["embedding"](uid, ids)
        out_bm25.extend(b[a] for a in ids)
        out_emb.extend(e[a] for a in ids)
    return out_bm25, out_emb


def assemble_features(feat: pl.DataFrame, bm25: list, emb: list, has_session_data: bool) -> np.ndarray:
    """The booster's design matrix for one chunk. The impute applies only
    where the feature existed at training time (EB-NeRD); on MIND it was null
    for every training row and stays null, exactly as the booster saw it."""
    clicks_earlier = pl.col("clicks_earlier_in_session")
    if has_session_data:
        clicks_earlier = clicks_earlier.fill_null(CLICKS_EARLIER_IMPUTE)
    frame = feat.with_columns(
        pl.Series("bm25_score", bm25, dtype=pl.Float64),
        pl.Series("embedding_score", emb, dtype=pl.Float64),
        pl.lit(False).alias("in_bm25_top200"),
        pl.lit(False).alias("in_embedding_top200"),
        clicks_earlier,
    )
    return feature_matrix(frame)


def ranks_per_impression(scores: np.ndarray, lengths: np.ndarray) -> list:
    out = []
    offset = 0
    for n in lengths:
        s = scores[offset:offset + n]
        out.append((np.argsort(np.argsort(-s, kind="stable"), kind="stable") + 1).tolist())
        offset += n
    assert offset == len(scores)
    return out


def rerank_population(dataset: str) -> Path:
    """`(impression_id, user_id, ranks, score_range)` for every impression, in
    user-sorted order, as one parquet; per-chunk checkpoints under
    CHECKPOINT_DIR/{dataset}/chunks."""
    final = CHECKPOINT_DIR / dataset / "reranker_ranks.parquet"
    if final.exists():
        log_progress(f"  {dataset}: ranks loaded from checkpoint")
        return final

    plan = chunk_plans[dataset]
    chunk_dir = CHECKPOINT_DIR / dataset / "chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    n_chunks = len(plan["imp_bounds"]) - 1
    booster = store[dataset]["booster"]
    log_progress(f"  {dataset}: re-ranking {store[dataset]['n_impressions']} impressions in {n_chunks} chunks")

    for c in range(n_chunks):
        chunk_path = chunk_dir / f"chunk_{c:03d}.parquet"
        if chunk_path.exists():
            continue
        scorers = get_scorers(dataset)  # built lazily, so a fully-checkpointed run never pays for it
        s, e = plan["imp_bounds"][c], plan["imp_bounds"][c + 1]
        r0, r1 = plan["row_bounds"][c], plan["row_bounds"][c + 1]

        src = pl.scan_parquet(plan["source"]).slice(s, e - s).collect()
        feat = pl.scan_parquet(store[dataset]["features_path"]).slice(r0, r1 - r0).collect()
        imp_ids = src["impression_id"].to_list()
        user_ids = src["user_id"].to_list()
        inviews = src["article_ids_inview"].to_list()
        lengths = np.fromiter((len(x) for x in inviews), dtype=np.int64, count=len(inviews))

        # Alignment check, every chunk: same impressions, same candidates,
        # same order on both sides. Cheap relative to the scoring below.
        for col, values in (("impression_id", imp_ids), ("user_id", user_ids)):
            expected = np.repeat(np.asarray(values, dtype=object), lengths)
            if feat.height != int(lengths.sum()) or not np.array_equal(feat[col].to_numpy(), expected):
                raise ValueError(f"{dataset}: chunk {c} feature rows are not aligned with the sorted source on {col}")
        expected_aid = [a for ids in inviews for a in ids]
        if feat["article_id"].to_list() != expected_aid:
            raise ValueError(f"{dataset}: chunk {c} candidate order differs between features and source")
        del expected, expected_aid

        bm25, emb = stage1_columns(scorers, user_ids, inviews)
        X = assemble_features(feat, bm25, emb, store[dataset]["has_session_data"])
        scores = booster.predict(X)
        ranks = ranks_per_impression(scores, lengths)

        # score_range: max-min of the booster's scores within the impression.
        # Persisted so the format test can show ties are rare (a constant
        # score would still be a valid permutation, just a meaningless one).
        offsets = np.concatenate([[0], np.cumsum(lengths)])
        score_range = [float(scores[a:b].max() - scores[a:b].min()) for a, b in zip(offsets[:-1], offsets[1:])]

        tmp = chunk_path.with_suffix(".parquet.tmp")
        pl.DataFrame({"impression_id": imp_ids, "user_id": user_ids, "ranks": ranks, "score_range": score_range}).write_parquet(tmp)
        os.replace(tmp, chunk_path)
        log_progress(f"  {dataset}: chunk {c + 1}/{n_chunks} checkpointed (impressions {s}-{e}, {feat.height} rows)")
        del src, feat, imp_ids, user_ids, inviews, bm25, emb, X, scores, ranks

    sink_parquet_atomic(pl.concat([pl.scan_parquet(chunk_dir / f"chunk_{c:03d}.parquet") for c in range(n_chunks)]), final)
    shutil.rmtree(chunk_dir)
    log_progress(f"  {dataset}: all {n_chunks} chunks merged into {final.name}")
    return final


rank_paths = {name: rerank_population(name) for name in DATASETS}
{name: pl.scan_parquet(p).select(pl.len()).collect().item() for name, p in rank_paths.items()}

{'mind_large_test': 2370727}

In [6]:
def test_feature_assembly():
    for name in DATASETS:
        plan = chunk_plans[name]
        booster = store[name]["booster"]
        s, e = 0, min(2_000, plan["imp_bounds"][-1])
        src = pl.scan_parquet(plan["source"]).slice(s, e - s).collect()
        lengths = src["article_ids_inview"].list.len().to_numpy().astype(np.int64)
        r1 = int(lengths.sum())
        feat = pl.scan_parquet(store[name]["features_path"]).slice(0, r1).collect()
        scorers = get_scorers(name)
        bm25, emb = stage1_columns(scorers, src["user_id"].to_list(), src["article_ids_inview"].to_list())
        assert len(bm25) == len(emb) == feat.height
        assert all(np.isfinite(bm25)) and all(np.isfinite(emb))

        X = assemble_features(feat, bm25, emb, store[name]["has_session_data"])
        assert X.shape == (feat.height, len(FEATURE_COLUMNS)) and X.dtype == np.float32
        j_bm25, j_emb = FEATURE_COLUMNS.index("bm25_score"), FEATURE_COLUMNS.index("embedding_score")
        assert np.allclose(X[:, j_bm25], np.asarray(bm25, dtype=np.float32))
        assert np.allclose(X[:, j_emb], np.asarray(emb, dtype=np.float32))
        for flag in ("in_bm25_top200", "in_embedding_top200"):
            assert (X[:, FEATURE_COLUMNS.index(flag)] == 0.0).all()
        j_ces = FEATURE_COLUMNS.index("clicks_earlier_in_session")
        if store[name]["has_session_data"]:
            assert (X[:, j_ces] == CLICKS_EARLIER_IMPUTE).all()
        else:
            assert np.isnan(X[:, j_ces]).all()  # MIND: null at training time too
        # Imputing 0 and leaving the value missing are prediction-identical on
        # both boosters: LightGBM substitutes 0 for NaN in a feature that had
        # no missing values at training time (EB-NeRD), and the MIND booster
        # has no split on the feature at all. So the impute documents rather
        # than changes what the booster does.
        Xn = X.copy(); Xn[:, j_ces] = np.nan
        Xz = X.copy(); Xz[:, j_ces] = CLICKS_EARLIER_IMPUTE
        assert np.array_equal(booster.predict(Xn), booster.predict(Xz))

        # Ranks: a valid permutation per impression, and the top-ranked
        # candidate is the argmax score.
        scores = booster.predict(X)
        ranks = ranks_per_impression(scores, lengths)
        off = 0
        for n, r in zip(lengths, ranks):
            assert sorted(r) == list(range(1, n + 1))
            assert r[int(np.argmax(scores[off:off + n]))] == 1
            off += n

        # Persisted ranks for these impressions equal a fresh recomputation.
        persisted = pl.scan_parquet(rank_paths[name]).slice(0, e - s).collect()
        assert persisted["impression_id"].to_list() == src["impression_id"].to_list()
        assert persisted["user_id"].to_list() == src["user_id"].to_list()
        assert persisted["ranks"].to_list() == ranks


test_feature_assembly()
print("ok: stage-1 scores land in the right columns, membership flags are False, the clicks_earlier imputation is prediction-identical to NaN, ranks are valid permutations with argmax at rank 1, and persisted ranks reproduce")

ok: stage-1 scores land in the right columns, membership flags are False, the clicks_earlier imputation is prediction-identical to NaN, ranks are valid permutations with argmax at rank 1, and persisted ranks reproduce


## Write the submission files

Ranks are re-ordered into `behaviors.parquet`'s original row order (the order
the leaderboards expect -- A1's notebooks established it) by a lazy join on
`(impression_id, user_id)`, then streamed out `WRITE_ROWS` lines at a time. Line format,
archive name and member name are byte-identical to A1's submissions:
EB-NeRD `"<native impression id> [r1,r2,...]"` in `predictions.txt`, MIND
`"<impression id> [r1,r2,...]"` in `prediction.txt`.

In [7]:
def write_submission(dataset: str) -> Path:
    s = store[dataset]
    ordered = CHECKPOINT_DIR / dataset / "reranker_ranks_ordered.parquet"
    if not ordered.exists():
        # The frame that gets sorted is as narrow as it can be: a row index
        # and the ranks as UInt16 (the widest in-view list is a few hundred).
        # Impression ids are re-read from behaviors in the write loop below.
        beh = pl.scan_parquet(s["behaviors_path"]).select(ROW_KEY).with_row_index("row_idx")
        sink_parquet_atomic(
            beh.join(pl.scan_parquet(rank_paths[dataset]).select(*ROW_KEY, "ranks"), on=ROW_KEY, how="left")
               .select("row_idx", pl.col("ranks").cast(pl.List(pl.UInt16)))
               .sort("row_idx")
               .select("ranks"),
            ordered,
        )
    n_rows = pl.scan_parquet(ordered).select(pl.len()).collect().item()
    n_missing = pl.scan_parquet(ordered).select(pl.col("ranks").null_count()).collect().item()
    if n_rows != s["n_impressions"] or n_missing:
        raise ValueError(f"{dataset}: ordered ranks have {n_rows} rows ({n_missing} without ranks), expected {s['n_impressions']}")

    txt_path = SUBMISSIONS_DIR / dataset / s["prediction_filename"]
    zip_path = SUBMISSIONS_DIR / dataset / f"{dataset}_reranker_predictions.zip"
    strip = s["strip_impression_prefix"]
    n_lines = 0
    # Bytes, not text mode: write_text would turn \n into \r\n on Windows.
    with txt_path.open("wb") as out:
        for start in range(0, n_rows, WRITE_ROWS):
            part = pl.scan_parquet(ordered).slice(start, WRITE_ROWS).collect()
            ids = pl.scan_parquet(s["behaviors_path"]).select("impression_id").slice(start, WRITE_ROWS).collect()["impression_id"].to_list()
            ranks = part["ranks"].to_list()
            assert len(ids) == len(ranks)
            lines = []
            for iid, r in zip(ids, ranks):
                native = iid.rsplit("_", 1)[-1] if strip else iid
                lines.append(f"{native} [{','.join(map(str, r))}]")
            out.write(("\n".join(lines) + "\n").encode("utf-8"))
            n_lines += len(lines)
            del part, ids, ranks, lines
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(txt_path, arcname=s["prediction_filename"])
    txt_path.unlink()
    log_progress(f"  {dataset}: wrote {zip_path.name} ({n_lines} lines)")
    return zip_path


submission_zips = {name: write_submission(name) for name in DATASETS}
log_progress("reranker_submission: all zips written")
submission_zips

{'mind_large_test': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/mind_large_test/mind_large_test_reranker_predictions.zip')}

In [8]:
def test_submission_files():
    for name in DATASETS:
        s = store[name]
        zip_path = submission_zips[name]
        # ebnerd_testset: 200,000 of these lines carry the id "0" (the
        # beyond-accuracy sentinel); Codabench matches them by position, and
        # A1's accepted submissions had exactly the same 200,000 lines.
        expected_ids = pl.scan_parquet(s["behaviors_path"]).select("impression_id").collect()["impression_id"].to_list()
        expected_len = pl.scan_parquet(s["behaviors_path"]).select(pl.col("article_ids_inview").list.len()).collect()["article_ids_inview"].to_list()
        if s["strip_impression_prefix"]:
            expected_ids = [i.rsplit("_", 1)[-1] for i in expected_ids]

        with zipfile.ZipFile(zip_path) as zf:
            assert zf.namelist() == [s["prediction_filename"]], zf.namelist()
            n = 0
            # Streamed line by line -- the EB-NeRD file is 13.5M lines.
            with zf.open(s["prediction_filename"]) as fh:
                for line in fh:
                    line = line.decode("utf-8").rstrip("\n")
                    assert not line.endswith("\r")
                    iid, ranks_str = line.split(" ", 1)
                    assert iid == expected_ids[n], (name, n, iid, expected_ids[n])
                    ranks = [int(r) for r in ranks_str.strip("[]").split(",")]
                    assert len(ranks) == expected_len[n]
                    if n % 997 == 0 or len(ranks) > 100:
                        assert sorted(ranks) == list(range(1, len(ranks) + 1))
                    n += 1
        assert n == len(expected_ids) == s["n_impressions"], (name, n)

        # Ties: the booster's scores are not constant within an impression in
        # essentially every case (an all-tie impression would make the rank
        # order the input order -- valid, but not a prediction).
        rng = pl.scan_parquet(rank_paths[name]).select(
            (pl.col("score_range") > 0).mean().alias("share_non_constant"), pl.len()
        ).collect().row(0)
        assert rng[0] > 0.99, (name, rng)
        print(f"  {name}: {n} lines, share of impressions with a non-constant score {rng[0]:.4f}")


test_submission_files()
log_progress("reranker_submission completed successfully")
print("ok: each zip holds exactly the expected member, one line per impression in behaviors order, correct lengths, valid permutations, LF endings, and non-constant scores")

  mind_large_test: 2370727 lines, share of impressions with a non-constant score 0.9980
ok: each zip holds exactly the expected member, one line per impression in behaviors order, correct lengths, valid permutations, LF endings, and non-constant scores


# Manual Review Complete

Upload `submissions/ebnerd_testset/ebnerd_testset_reranker_predictions.zip` to
the RecSys 2024 Challenge (Codabench 2469) and
`submissions/mind_large_test/mind_large_test_reranker_predictions.zip` to the
MIND competition (Codabench 13967). Screenshots go in the design note (Q5).